# L6.2 · Flow matching: el camino recto y la brújula

| Notebook | Qué abre |
|---|---|
| L6.1 | El problema: por qué la regresión cae en el medio, y por qué tampoco sirve aprender "ruido → acción" de un salto |
| **L6.2** | **El camino recto y la brújula: qué velocidad aprende la red y cómo se entrena** |
| L6.3 | La inferencia: caminar desde el ruido de a pasitos |
| L6.4 | Con datos reales del G1: bloques de 48 acciones × 26 articulaciones |
| L6.5 | El código real de LeRobot (SmolVLA, π0), lado a lado con el nuestro |

En L6.1 vimos que aprender el salto "ruido → demostración" con parejas al azar
cae en el promedio, porque la red no puede saber qué ruido va con qué
demostración. Acá cambiamos la pregunta: en vez del salto entero, cada pasito.

Convención: `t` es **cuánto del camino recorriste**. `t = 0` es ruido puro y
`t = 1` es una demostración. (LeRobot lo escribe al revés, con `t = 1` como
ruido; es la misma cuenta dada vuelta y lo vemos en L6.5.)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

from toolkit.flow import toy

plt.rcParams["figure.dpi"] = 110
torch.set_num_threads(4)  # redes chiquitas: más núcleos sólo agregan coordinación
torch.manual_seed(0)

demos = toy.sample_demos(400)


def wide_view(ax):
    """La escena con lugar para el ruido, que vive alrededor del robot."""
    ax.set_xlim(-2.6, 2.6)
    ax.set_ylim(-2.6, 3.0)

## 1. El camino: una línea recta de ruido a demostración

Agarramos una pareja cualquiera: un punto de ruido `x0` (sorteado de una
campana alrededor del robot) y una demostración `x1`. El camino más simple
entre los dos es la recta:

$$x_t = (1 - t)\,x_0 + t\,x_1$$

En `t = 0` estás en el ruido, en `t = 1` en la demostración, en `t = 0.5` justo
a mitad de camino. Dibujamos 40 parejas al azar, con un punto cada cuarto del
camino:

In [ ]:
pairs = 40
x0 = torch.randn(pairs, 2)
x1 = demos[torch.randint(len(demos), (pairs,))]

fig, ax = plt.subplots(figsize=(6.5, 6))
toy.draw_scene(ax, title="40 parejas (ruido, demostración) unidas por rectas", alpha=0.15)
wide_view(ax)
colors = toy.side_colors(x1)
for a, b, color in zip(x0, x1, colors):
    ax.plot([a[0], b[0]], [a[1], b[1]], color=color, lw=0.8, alpha=0.6)
for t in [0.25, 0.5, 0.75]:
    xt = (1 - t) * x0 + t * x1
    ax.scatter(xt[:, 0], xt[:, 1], s=8, c=colors, zorder=3)
ax.scatter(x0[:, 0], x0[:, 1], s=14, color="gray", zorder=3, label="ruido (t = 0)")
ax.legend(loc="lower left", fontsize=8)

**Ojo: estas rectas no son caminos del robot.** El robot no camina desde
ningún lado: la acción es sólo el punto final, por dónde pasar al costado del
árbol. Las rectas son el camino que hace **el número** mientras lo generamos,
de ruido a acción. Que alguna cruce el árbol no importa; lo único que importa
es dónde termina. (En un robot de verdad, la "acción" son 26 ángulos y el
camino vive en un espacio de 26 dimensiones que ni siquiera se puede dibujar.)

Cada línea tiene el color de su destino: azul si termina a la izquierda del
árbol, naranja si a la derecha. Fijate que **el ruido no "sabe" adónde va**:
puntos grises vecinos pueden terminar en lados opuestos, y las líneas se
cruzan. Eso va a importar en un rato.

Ahora miramos la nube entera (miles de parejas) en cinco momentos del camino,
con el mismo color por destino:

In [ ]:
many = 3000
x0_many = torch.randn(many, 2)
x1_many = demos[torch.randint(len(demos), (many,))]
colors_many = toy.side_colors(x1_many)

fig, axes = plt.subplots(1, 5, figsize=(17, 4))
for ax, t in zip(axes, [0.0, 0.25, 0.5, 0.75, 1.0]):
    xt = (1 - t) * x0_many + t * x1_many
    toy.draw_scene(ax, title=f"t = {t}")
    wide_view(ax)
    ax.scatter(xt[:, 0], xt[:, 1], s=2, c=colors_many, alpha=0.5)
plt.tight_layout()

Al principio (`t = 0`) los colores están mezclados: el ruido no sabe adónde
va. A mitad de camino ya se separan en dos nubes que se tocan en el medio. Al
final, cada color llegó a su grupo y nadie quedó en el árbol.

Pero acá hicimos trampa: cada punto sigue la recta **de su pareja**, que
elegimos nosotros. El truco va a ser lograr lo mismo sin conocer la pareja.

## 2. La velocidad de cada pareja

Si caminás por la recta de `x0` a `x1` en un tiempo total de 1, tu velocidad
es constante: en cada instante te movés `x1 - x0` por unidad de tiempo. Para
cada pareja, entonces, hay una respuesta exacta a la pregunta "estoy en `x_t`,
¿para dónde me muevo?":

$$\text{velocidad de la pareja} = x_1 - x_0$$

Eso es lo que la red va a aprender a predecir: le damos el punto `x_t` y el
momento `t`, y tiene que decir la velocidad. Una **brújula**: no dice adónde
vas a llegar, dice para dónde moverte desde acá.

## 3. El problema: las líneas se cruzan

Pero la red no ve la pareja: ve sólo **el punto y el momento**. Y por un mismo
punto, en un mismo momento, pasan muchas líneas con velocidades distintas.
Miramos dos puntos en `t = 0.5`: uno justo en el medio y otro un poco a la
izquierda. Buscamos, entre 200.000 parejas, las que pasan cerca de cada uno:

In [ ]:
t_check = 0.5
x0_big = torch.randn(200_000, 2)
x1_big = demos[torch.randint(len(demos), (200_000,))]
xt_big = (1 - t_check) * x0_big + t_check * x1_big

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, point in zip(axes, [(0.0, 1.0), (-0.35, 1.0)]):
    label = f"({point[0]}, {point[1]})"
    point = torch.tensor(point)
    near = torch.linalg.vector_norm(xt_big - point, dim=1) < 0.06
    velocities = (x1_big - x0_big)[near]
    toy.draw_scene(ax, title=f"{int(near.sum())} parejas pasan cerca de {label} en t = 0.5", alpha=0.1)
    wide_view(ax)
    for a, b in zip(x0_big[near][:60], x1_big[near][:60]):
        ax.plot([a[0], b[0]], [a[1], b[1]], color=toy.side_colors(b[None])[0], lw=0.6, alpha=0.4)
    scale = 0.35
    for v, b in zip(velocities[:60], x1_big[near][:60]):
        ax.arrow(point[0], point[1], scale * v[0], scale * v[1], width=0.006, color=toy.side_colors(b[None])[0], alpha=0.5)
    mean_v = velocities.mean(0)
    ax.arrow(point[0], point[1], scale * mean_v[0], scale * mean_v[1], width=0.025, color="red", zorder=5,
             label="el promedio: lo que aprende la red")
    ax.legend(loc="lower left", fontsize=8)
    left = (x1_big[near][:, 0] < 0).float().mean()
    print(f"punto {label}: {left:.0%} de las parejas van a la izquierda · velocidad promedio {mean_v.numpy().round(2)}")
plt.tight_layout()

En el punto del medio, la mitad de las parejas va a la izquierda y la mitad a
la derecha, y el promedio apunta derecho al tronco. ¿Volvimos al problema de
L6.1? No del todo. En el punto un poco a la izquierda, el 79 % va a la
izquierda y la flecha promedio ya se inclina hacia ese lado. Y una vez que te
moviste un poco a la izquierda, la proporción sube más y la flecha se inclina
más: **la decisión se amplifica sola a lo largo del camino**. El medio exacto
es una línea muy fina, y casi ningún ruido cae justo ahí.

Ese promedio por punto y por momento tiene una cuenta exacta. Con nuestras 400
demostraciones se puede calcular a mano, sin red, y eso nos da **la brújula
ideal** contra la cual comparar lo que aprenda la red.

## 4. La brújula ideal, calculada a mano

Si estoy en `x` en el momento `t`, ¿de qué demostración vengo, probablemente?
Cada demostración `x1` "explica" mi posición si `x` quedó cerca de `t · x1`
(lo que aporta la demostración a esta altura del camino), con un margen que es
el ruido que queda, `1 - t`. Le doy a cada demostración un peso según cuán bien
explica `x`, promedio las demostraciones con esos pesos, y la brújula apunta
hacia ese promedio:

$$v^*(x, t) = \frac{\text{promedio pesado de las demostraciones} - x}{1 - t}$$

(Dividir por `1 - t` es porque queda `1 - t` de tiempo para llegar.)

In [ ]:
def ideal_velocity(x: torch.Tensor, t: float, demos: torch.Tensor) -> torch.Tensor:
    """El promedio exacto de las velocidades de todas las parejas que pasan por x en el momento t."""
    remaining = 1.0 - t  # el ruido que queda: el margen con que cada demostración explica x
    squared_distance = ((x[:, None, :] - t * demos[None, :, :]) ** 2).sum(-1)
    weights = torch.softmax(-squared_distance / (2 * remaining**2), dim=1)  # cuánto explica x cada demostración
    expected_destination = weights @ demos
    return (expected_destination - x) / remaining


grid_x, grid_y = torch.meshgrid(torch.linspace(-2.4, 2.4, 17), torch.linspace(-2.4, 2.8, 17), indexing="xy")
grid = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1)


def draw_field(ax, velocity, color="black", scale=None):
    """Una flecha por punto de la grilla; el largo se normaliza para que se lean todas."""
    v = velocity / (torch.linalg.vector_norm(velocity, dim=1, keepdim=True) + 1e-8)
    ax.quiver(grid[:, 0], grid[:, 1], v[:, 0], v[:, 1], color=color, scale=scale or 28, width=0.004, alpha=0.8)


fig, axes = plt.subplots(1, 5, figsize=(17, 4))
for ax, t in zip(axes, [0.0, 0.25, 0.5, 0.75, 0.95]):
    xt = (1 - t) * x0_many + t * x1_many
    toy.draw_scene(ax, title=f"brújula ideal en t = {t}")
    wide_view(ax)
    ax.scatter(xt[:, 0], xt[:, 1], s=1, c=colors_many, alpha=0.25)
    draw_field(ax, ideal_velocity(grid, t, demos))
plt.tight_layout()

Todas las flechas tienen el mismo largo: sólo muestran la dirección.

Al principio (`t = 0`) **todas apuntan al tronco**: sin información, la mejor
apuesta es el promedio, lo mismo que en L6.1. Pero eso es sólo el primer
pasito. A medida que avanza el camino el campo se parte: del lado izquierdo
las flechas apuntan al grupo de la izquierda, y del lado derecho al de la
derecha. Cerca del final, cada flecha apunta al grupo más cercano.

Las flechas lejos de la nube no importan: en ese momento del camino, ningún
punto pasa por ahí.

## 5. Entrenar la red para que sea la brújula

Ahora sí, el entrenamiento de flow matching, entero. En cada paso:

1. Sorteo demostraciones `x1` y ruido `x0`, emparejados al azar (como en
   L6.1, que antes no funcionaba).
2. Sorteo un momento `t` para cada pareja, entre 0 y 1.
3. Calculo el punto del camino: `x_t = (1 - t) x0 + t x1`.
4. Le pido a la red la velocidad en `(x_t, t)` y la comparo con la de la
   pareja, `x1 - x0`, con error cuadrático.

Eso es todo. La red recibe el punto (2 números) y el momento (1 número), y
devuelve una velocidad (2 números). En los modelos de verdad el momento entra
de una forma más rica (L6.5); acá alcanza con pegarlo como un número más.

In [ ]:
class VelocityNet(nn.Module):
    """La brújula: (punto, momento) → velocidad."""

    def __init__(self, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))


def flow_matching_step(model, demos, batch_size=256):
    x1 = demos[torch.randint(len(demos), (batch_size,))]  # 1. demostraciones...
    x0 = torch.randn_like(x1)                              #    ...y ruido, emparejados al azar
    t = torch.rand(batch_size, 1)                          # 2. un momento por pareja
    xt = (1 - t) * x0 + t * x1                             # 3. el punto del camino
    target = x1 - x0                                       # 4. la velocidad de la pareja
    return ((model(xt, t) - target) ** 2).mean()


torch.manual_seed(0)
model = VelocityNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
snapshot_steps = [0, 100, 500, 4000]
snapshots, losses = {}, []
for step in range(4001):
    if step in snapshot_steps:
        with torch.no_grad():
            snapshots[step] = model(grid, torch.full((len(grid), 1), 0.5))
    loss = flow_matching_step(model, demos)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print(f"pérdida al final (promedio de los últimos 500 pasos): {np.mean(losses[-500:]):.3f}")

### La pérdida no llega a cero, y está bien

Si las líneas se cruzan, ninguna red puede acertar la velocidad de cada pareja:
en el mismo punto y momento hay parejas que van para lados distintos. Hay un
**piso**: el error que queda aun respondiendo perfecto, o sea respondiendo la
brújula ideal. Lo medimos con la brújula de la sección 4 y lo comparamos con la
curva de entrenamiento:

In [ ]:
with torch.no_grad():
    x1 = demos[torch.randint(len(demos), (20_000,))]
    x0 = torch.randn_like(x1)
    t = torch.rand(20_000, 1)
    xt = (1 - t) * x0 + t * x1
    ideal = torch.cat([ideal_velocity(xt[i:i + 1], t[i].item(), demos) for i in range(len(xt))])
    floor = ((ideal - (x1 - x0)) ** 2).mean().item()

smooth = np.convolve(losses, np.ones(50) / 50, mode="valid")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(smooth, color="red", label="pérdida de la red (promedio móvil)")
ax.axhline(floor, color="black", ls="--", label=f"el piso: la brújula ideal ({floor:.3f})")
ax.set_xlabel("paso de entrenamiento")
ax.set_ylabel("error cuadrático")
ax.set_ylim(0, max(3, floor * 2))
ax.legend()
print(f"piso {floor:.3f} · red {np.mean(losses[-500:]):.3f}")

La red llega muy cerca del piso: aprendió casi todo lo que se podía
aprender. El resto de la pérdida no tiene arreglo: es la parte de "¿a qué lado
iba esta pareja?" que el punto y el momento no alcanzan a decir. Por eso en
flow matching la curva de pérdida engaña si se la mira sola: nunca baja a
cero, y que no baje más no quiere decir que la red ande mal.

## 6. Fotos de cómo aprende

La brújula de la red en `t = 0.5`, en cuatro momentos del entrenamiento (en
rojo), sobre la ideal (en gris):

In [ ]:
ideal_half = ideal_velocity(grid, 0.5, demos)
xt_half = 0.5 * x0_many + 0.5 * x1_many
fig, axes = plt.subplots(1, 4, figsize=(17, 4.3))
for ax, step in zip(axes, snapshot_steps):
    toy.draw_scene(ax, title=f"t = 0.5 · después de {step} pasos")
    wide_view(ax)
    ax.scatter(xt_half[:, 0], xt_half[:, 1], s=1, c=colors_many, alpha=0.2)
    draw_field(ax, ideal_half, color="gray")
    draw_field(ax, snapshots[step], color="red")
plt.tight_layout()

Al principio la red dice cualquier cosa. A los 100 pasos ya aprendió lo grueso
("para arriba, hacia el árbol"). Lo fino, cómo se abren las flechas alrededor
del árbol, tarda más.

Y la red ya entrenada contra la ideal, en todo el camino. El error lo medimos
**donde está la nube** en cada momento (los puntos por los que de verdad pasan
caminos), como porcentaje del largo de la flecha ideal:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.3))
for ax, t in zip(axes, [0.1, 0.4, 0.7, 0.9]):
    cloud = (1 - t) * x0_many[:500] + t * x1_many[:500]
    with torch.no_grad():
        on_cloud = model(cloud, torch.full((len(cloud), 1), t))
        learned = model(grid, torch.full((len(grid), 1), t))
    ideal_cloud = ideal_velocity(cloud, t, demos)
    error = (torch.linalg.vector_norm(on_cloud - ideal_cloud, dim=1) / torch.linalg.vector_norm(ideal_cloud, dim=1)).median()
    toy.draw_scene(ax, title=f"t = {t} · error típico en la nube {error:.0%}")
    wide_view(ax)
    ax.scatter(cloud[:, 0], cloud[:, 1], s=2, c=toy.side_colors(x1_many[:500]), alpha=0.4)
    draw_field(ax, ideal_velocity(grid, t, demos), color="gray")
    draw_field(ax, learned, color="red")
plt.tight_layout()

Donde pasan los caminos, la red se equivoca en un 4 a 6 % del largo de la
flecha ideal. Lejos de la nube (arriba del árbol, cerca del final) las flechas
rojas y grises no coinciden, y no importa: ahí no pasa ningún camino, así que
la red nunca vio un ejemplo de esa zona y nunca lo va a necesitar.

## Lo que nos llevamos

- El camino de cada pareja es una recta, y su velocidad es constante:
  `x1 - x0`. Eso es lo que se le pide a la red.
- La red no ve la pareja, sólo el punto y el momento. Por un mismo punto pasan
  parejas que van a lados distintos, así que aprende **el promedio de las
  velocidades** que pasan por ahí: la brújula.
- Promediar destinos (L6.1) da el tronco. Promediar direcciones da un campo de
  flechas que, al principio, apunta al medio y, a medida que avanza el camino,
  se parte en dos.
- La pérdida no baja a cero: tiene un piso que es la parte que no se puede
  adivinar. Que la red llegue al piso quiere decir que aprendió todo lo que se
  podía aprender.

Falta lo más importante: ver que, **caminando** con esta brújula desde el ruido,
se llega a demostraciones válidas, con las proporciones correctas, y nunca al
tronco. Eso es L6.3.

## Referencias

- Lipman et al., [Flow Matching for Generative Modeling](https://arxiv.org/abs/2210.02747)
  (2023): la pérdida de esta sección, y la prueba de que aprender el promedio
  de las velocidades de las parejas alcanza.
- Liu et al., [Flow Straight and Fast](https://arxiv.org/abs/2209.03003)
  (2023): el camino recto entre parejas al azar (*rectified flow*).
- Lipman et al., [Flow Matching Guide and Code](https://arxiv.org/abs/2412.06264)
  (2024): la brújula ideal de la sección 4 es su "campo de velocidad marginal".